# Diagnóstico dos marcadores aprendidos

**Pergunta:** por que o pipeline segmenta mais que o GT (precisão baixa), se os marcadores aprendidos caem sobre os núcleos?
Com marcadores binários de área parecida tirados do GT (o "miolo"), o ScribblePrompt segmenta **menos** que o GT.

Treina o experimento-base (soft Dice + TV, ScribblePrompt em 256² no modo `positive`) com a mesma validação cruzada de 5
*folds*, guarda o modelo da melhor época de cada *fold* e, **sem retreinar**, testa três explicações nas 37 imagens de
treino, cada uma prevista pelo modelo do *fold* em que ela ficou de fora:

| | Hipótese | Teste |
|---|---|---|
| H1 | A probabilidade de saída fica um pouco acima de 0,5 em volta dos núcleos | limiar de saída de 0,5 a 0,9 |
| H2 | O formato do marcador aprendido (suave, alongado) faz a rede expandir mais | o mesmo marcador **binarizado**, e o miolo do GT nas mesmas imagens |
| H3 | Sem canal negativo, nada impede núcleos vizinhos de se fundirem | o marcador aprendido com o fundo do Cellpose (a mais de 3 px das máscaras) como negativo |

Também mede onde cai o marcador, onde ficam os falsos positivos e quantos núcleos a saída funde.

**Tudo é mostrado no notebook.** Salvar os resultados e os modelos é opcional (`SALVAR_RESULTADOS`, `SALVAR_MODELOS`).
Use um ambiente com **GPU** (~40 min numa T4, a maior parte no treino dos 5 *folds*).

## 0. Setup

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

In [ ]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

## 1. Configuração

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.patches import Patch
from scipy import ndimage
from skimage.segmentation import find_boundaries

from src.data.load.preprocessed_dataset import PreprocessedDataset, collate_samples, load_preprocessed
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import provenance
from src.losses.loss_composer import LossComposer
from src.losses.terms import DiceTerm, TVTerm
from src.models.configs.marker_unet_config import UNET_CONFIG
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.models.networks.marker_unet import MarkerUNet
from src.pipeline.steps.inference.frozen_segmentation_step import FrozenSegmentationStep
from src.pipeline.steps.inference.marker_step import MarkerStep
from src.pipeline.training_pipeline import TrainingPipeline
from src.training.kfold import TrainConfig, run_kfold

NOME = "diagnostico_marcadores"
SALVAR_RESULTADOS = False    # True: grava tabelas, figura e config em docs/estudo/resultados/<NOME>/
SALVAR_MODELOS = False       # True: grava o modelo de cada fold em MODELS_DIR/<NOME>/ (~100 MB cada)
OUTPUT_DIR = os.path.join("docs", "estudo", "resultados", NOME)
MODELS_DIR = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))

# Mesma configuração do experimento-base: mesmas partições, sementes e curva de lr.
RGBA_ALPHA = "mask"
SP_INPUT_SIZE = (256, 256)
TAU = 0.5
TV_WEIGHT = 0.001
CONFIG = TrainConfig(
    k=5, batch_size=4, max_epochs=200, patience=20, min_delta=0.0,
    lr=1e-4, grad_clip=1.0, threshold=0.5, seed=42, log_every=10,
)

LIMIARES = [0.5, 0.6, 0.7, 0.8, 0.9]
RAIO_NEGATIVO = 3            # negativo = pixels a mais de 3 px de qualquer máscara do Cellpose
IMAGENS_20X = {"TCGA-HE-7128-01Z-00-DX1", "TCGA-HE-7129-01Z-00-DX1", "TCGA-HE-7130-01Z-00-DX1"}

print("Dispositivo:", CONFIG.device, "" if torch.cuda.is_available() else "(sem GPU: o treino fica muito lento)")

## 2. Dados

In [ ]:
train_samples = load_preprocessed("train", alpha=RGBA_ALPHA)
TRAIN_IDS = sorted(train_samples)
print(f"Treino: {len(train_samples)} imagens")


def disco(raio):
    y, x = np.ogrid[-raio:raio + 1, -raio:raio + 1]
    return x * x + y * y <= raio * raio


def dilatar(mascara, raio):
    return ndimage.binary_dilation(mascara, structure=disco(raio)) if raio > 0 else mascara

## 3. Treino (validação cruzada)

A função que monta o experimento é chamada uma vez por *fold*; cada modelo criado fica guardado em `MODELOS`. Ao fim de cada
*fold*, `run_kfold` carrega nele os pesos da melhor época.

In [ ]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_DIR, "scribbleprompt"))
FINAL_NET = ScribblePromptingNetwork(
    checkpoint=checkpoint, device=CONFIG.device, scribble_mode="positive",
    positive_threshold=TAU, input_size=SP_INPUT_SIZE,
)
MODELOS = []


def factory():
    marker_net = MarkerUNet(config=UNET_CONFIG)
    pipeline = TrainingPipeline([
        MarkerStep(model=marker_net, differentiable=True, device=CONFIG.device),
        FrozenSegmentationStep(final_network=FINAL_NET, device=CONFIG.device),
    ])
    MODELOS.append((marker_net, pipeline))
    return marker_net, pipeline, LossComposer([DiceTerm(), TVTerm(weight=TV_WEIGHT)])

In [ ]:
inicio = time.time()
folds = run_kfold(factory, train_samples, TRAIN_IDS, CONFIG)
assert len(MODELOS) == len(folds)
print(f"Treino: {(time.time() - inicio) / 60:.1f} min")

for f in folds:
    cp = np.mean([r["cellpose_dice"] for r in f.rows])
    print(f"fold {f.fold}: melhor época {f.best_epoch:3d} de {f.epochs_run:3d} | Dice val {f.best_dice:.4f} | Cellpose {cp:.4f}")

# Comparação com o experimento-base registrado (mesma configuração; a GPU não é determinística).
referencia = os.path.join("docs", "estudo", "resultados", "base_dice_tv", "folds.csv")
if os.path.exists(referencia):
    ref = pd.read_csv(referencia)
    agora = pd.DataFrame([r for f in folds for r in f.rows])
    print(f"Dice médio de validação: agora {agora.dice.mean():.4f} | base_dice_tv registrado {ref.dice.mean():.4f}")

if SALVAR_MODELOS:
    pasta = os.path.join(MODELS_DIR, NOME)
    os.makedirs(pasta, exist_ok=True)
    for f, (marker_net, _) in zip(folds, MODELOS):
        torch.save(marker_net.state_dict(), os.path.join(pasta, f"fold{f.fold}.pt"))
    print("Modelos em:", pasta)

## 4. Predições e condições

Para cada imagem, com o modelo do *fold* em que ela ficou de fora:

- `aprendido`: a saída do pipeline (marcador aprendido → canal positivo `relu(m − τ)/(1 − τ)`, negativo zero);
- `aprendido binário`: o mesmo canal positivo trocado por 1 onde ele é > 0;
- `aprendido + negativo` e `aprendido binário + negativo`: os mesmos, com o fundo do Cellpose como negativo;
- `miolo do GT`: pixels de núcleo com mapa de distância ≤ 0,5 (oráculo), sem negativo.

A primeira linha impressa confere que a chamada direta ao ScribblePrompt reproduz a saída do pipeline.

In [ ]:
def predizer(pipeline, image_id):
    """Marcador (sigmoide da MarkerUNet) e probabilidade de saída do pipeline, em 1000²."""
    for step in pipeline.steps:
        if hasattr(step, "set_training"):
            step.set_training(False)
    batch = collate_samples([PreprocessedDataset(train_samples, ids=[image_id])[0]])
    with torch.no_grad():
        out = pipeline.run({k: (v.to(CONFIG.device) if torch.is_tensor(v) else v) for k, v in batch.items()},
                           verbose=False)
    return out["markers"][0, 0].float().cpu().numpy(), out["segmentation"][0, 0].float().cpu().numpy()


def segmentar(image, positivo, negativo):
    scribbles = np.stack([positivo, negativo], axis=-1).astype(np.float32)
    with torch.no_grad():
        return FINAL_NET.predict({"image": image, "scribbles": scribbles})[0, 0].float().cpu().numpy()


PRED = {}
for f, (_, pipeline) in zip(folds, MODELOS):
    for image_id in f.val_ids:
        s = train_samples[image_id]
        marcador, prob = predizer(pipeline, image_id)
        pos = np.maximum(marcador - TAU, 0.0) / (1.0 - TAU)
        zero = np.zeros_like(pos)
        neg = (~dilatar(s["cellpose_segmentation"] > 0, RAIO_NEGATIVO)).astype(np.float32)
        binario = (pos > 0).astype(np.float32)
        miolo = ((s["ground_truth"] > 0) & (s["distance_map"] <= 0.5)).astype(np.float32)
        PRED[image_id] = {
            "fold": f.fold, "marcador": marcador.astype(np.float16), "pos": pos.astype(np.float16),
            "condicoes": {
                "aprendido": prob.astype(np.float16),
                "aprendido binário": segmentar(s["image"], binario, zero).astype(np.float16),
                "aprendido + negativo": segmentar(s["image"], pos, neg).astype(np.float16),
                "aprendido binário + negativo": segmentar(s["image"], binario, neg).astype(np.float16),
                "miolo do GT": segmentar(s["image"], miolo, zero).astype(np.float16),
            },
        }

primeira = next(iter(PRED))
s = train_samples[primeira]
direto = segmentar(s["image"], PRED[primeira]["pos"].astype(np.float32), np.zeros((1000, 1000), np.float32))
print("Diferença máxima entre a chamada direta e o pipeline:",
      float(np.abs(direto - PRED[primeira]["condicoes"]["aprendido"].astype(np.float32)).max()))

## 5. Métricas por condição e limiar (H1, H2, H3)

In [ ]:
linhas = []
for image_id, p in PRED.items():
    s = train_samples[image_id]
    gt = s["ground_truth"]
    base = binary_metrics(s["cellpose_segmentation"] > 0, gt)
    for condicao, prob in p["condicoes"].items():
        prob = prob.astype(np.float32)
        for limiar in LIMIARES:
            linhas.append({"id": image_id, "fold": p["fold"], "aumento_20x": image_id in IMAGENS_20X,
                           "condicao": condicao, "limiar": limiar, **binary_metrics(prob, gt, limiar),
                           **{f"cellpose_{k}": v for k, v in base.items()}})
df = pd.DataFrame(linhas)
df["dif_dice"] = df["dice"] - df["cellpose_dice"]

ORDEM = list(next(iter(PRED.values()))["condicoes"])
cellpose = df.drop_duplicates("id")
print("Cellpose:", cellpose[["cellpose_dice", "cellpose_precision", "cellpose_recall", "cellpose_mass_ratio"]]
      .mean().round(3).to_dict())
pd.set_option("display.width", 200)
resumo = df.groupby(["condicao", "limiar"]).agg(
    dice=("dice", "mean"), precisao=("precision", "mean"), revocacao=("recall", "mean"),
    massa=("mass_ratio", "mean"), dif_dice=("dif_dice", "mean"),
    n_melhor=("dif_dice", lambda d: int((d > 0).sum())),
).reset_index()
resumo["condicao"] = pd.Categorical(resumo["condicao"], ORDEM)
resumo = resumo.sort_values(["condicao", "limiar"])
resumo.round(3)

Dice por condição e limiar (linhas = condição, colunas = limiar):

In [ ]:
display(resumo.pivot(index="condicao", columns="limiar", values="dice").round(3))
resumo.pivot(index="condicao", columns="limiar", values="massa").round(2)

## 6. Onde cai o marcador e onde ficam os erros

Por imagem, com a saída `aprendido` no limiar 0,5. As regiões usam a distância de cada pixel ao GT. O GT é semântico: núcleos
que se tocam contam como um só componente.

- `marcador_*`: fração da massa do canal positivo dentro do GT, na faixa de até 3 px fora dele e mais longe;
- `nucleos_com_marcador`: fração dos componentes do GT tocados pelo marcador; `marcadores_por_nucleo`: componentes do marcador
  por componente do GT;
- `fp_*`: fração dos falsos positivos na faixa de até 3 px do GT (inchaço da borda), de 3 a 8 px e a mais de 8 px (manchas
  soltas), para a saída e para o Cellpose;
- `fusao_*`: fração dos componentes da saída que cobrem 2 ou mais componentes do GT.

In [ ]:
def faixas_fp(pred, gt, dist):
    fp = pred & ~gt
    total = max(int(fp.sum()), 1)
    return (fp & (dist <= 3)).sum() / total, (fp & (dist > 3) & (dist <= 8)).sum() / total, (fp & (dist > 8)).sum() / total


def fusao(pred, lab_gt):
    lab_p, n_p = ndimage.label(pred)
    sobre = pred & (lab_gt > 0)
    pares = np.unique(np.stack([lab_p[sobre], lab_gt[sobre]]), axis=1)
    por_componente = np.bincount(pares[0], minlength=n_p + 1)[1:]
    return float((por_componente >= 2).sum()) / max(n_p, 1)


diag = []
for image_id, p in PRED.items():
    s = train_samples[image_id]
    gt = s["ground_truth"] > 0
    cp = s["cellpose_segmentation"] > 0
    dist = ndimage.distance_transform_edt(~gt)
    pos = p["pos"].astype(np.float32)
    pred = p["condicoes"]["aprendido"].astype(np.float32) >= 0.5
    massa = max(float(pos.sum()), 1e-6)
    lab_gt, n_gt = ndimage.label(gt)
    tocados = np.unique(lab_gt[(pos > 0) & gt])
    _, n_m = ndimage.label(pos > 0)
    fp_s = faixas_fp(pred, gt, dist)
    fp_c = faixas_fp(cp, gt, dist)
    diag.append({
        "id": image_id, "aumento_20x": image_id in IMAGENS_20X,
        "marcador_area": float((pos > 0).mean()),
        "marcador_no_gt": pos[gt].sum() / massa,
        "marcador_borda_3px": pos[(dist > 0) & (dist <= 3)].sum() / massa,
        "marcador_longe": pos[dist > 3].sum() / massa,
        "nucleos_com_marcador": len(tocados[tocados > 0]) / max(n_gt, 1),
        "marcadores_por_nucleo": n_m / max(n_gt, 1),
        "fp_borda_3px": fp_s[0], "fp_3a8px": fp_s[1], "fp_longe": fp_s[2],
        "cellpose_fp_borda_3px": fp_c[0], "cellpose_fp_3a8px": fp_c[1], "cellpose_fp_longe": fp_c[2],
        "fusao_saida": fusao(pred, lab_gt), "fusao_cellpose": fusao(cp, lab_gt),
    })
diag = pd.DataFrame(diag)
diag.drop(columns=["id", "aumento_20x"]).describe().loc[["mean", "50%", "min", "max"]].T.round(3)

**Perfil da probabilidade em volta dos núcleos (H1).** Probabilidade média de saída por faixa de distância ao GT, para a saída
aprendida e para o miolo do GT. Se a saída aprendida tiver valores pouco acima de 0,5 nas faixas de 1 a 4 px, um limiar maior
corta o inchaço.

In [ ]:
FAIXAS = [("dentro do GT", lambda d: d == 0), ("1–2 px fora", lambda d: (d > 0) & (d <= 2)),
          ("3–4 px", lambda d: (d > 2) & (d <= 4)), ("5–8 px", lambda d: (d > 4) & (d <= 8)), ("> 8 px", lambda d: d > 8)]
perfil = []
for image_id, p in PRED.items():
    dist = ndimage.distance_transform_edt(~(train_samples[image_id]["ground_truth"] > 0))
    for condicao in ("aprendido", "miolo do GT"):
        prob = p["condicoes"][condicao].astype(np.float32)
        for nome, regra in FAIXAS:
            sel = regra(dist)
            perfil.append({"condicao": condicao, "faixa": nome, "prob_media": float(prob[sel].mean()),
                           "frac_acima_0.5": float((prob[sel] >= 0.5).mean())})
perfil = pd.DataFrame(perfil)
perfil["faixa"] = pd.Categorical(perfil["faixa"], [n for n, _ in FAIXAS])
display(perfil.pivot_table(index="faixa", columns="condicao", values="prob_media", observed=True).round(3))
perfil.pivot_table(index="faixa", columns="condicao", values="frac_acima_0.5", observed=True).round(3)

## 7. Imagens em 20× contra as demais

In [ ]:
sel = df[df.limiar == 0.5]
sel.groupby(["condicao", "aumento_20x"], observed=True)[["dice", "precision", "recall", "mass_ratio", "cellpose_dice"]] \
   .mean().reindex(ORDEM, level=0).round(3)

## 8. Figuras

Três imagens: a de pior, a mediana e a de melhor diferença de Dice contra o Cellpose (saída `aprendido`, limiar 0,5). Em cada
uma, um recorte de 250 × 250 px (a janela com mais núcleo, numa grade de 125 px). Painéis: imagem com o contorno do GT;
marcador da MarkerUNet; canal positivo que entra no ScribblePrompt; probabilidade de saída com o contorno do GT; mapas de erro
da saída aprendida, sem e com o negativo do Cellpose, e do Cellpose. Nos mapas de erro: acerto em cinza, falso positivo em
verde-água, falso negativo em violeta.

In [ ]:
COR_VP = np.array([0xB8, 0xB7, 0xB0]) / 255
COR_FP = np.array([0x1B, 0xAF, 0x7A]) / 255
COR_FN = np.array([0x4A, 0x3A, 0xA7]) / 255
LADO = 250


def janela(gt, lado=LADO, passo=125):
    h, w = gt.shape
    cantos = [(y, x) for y in range(0, h - lado + 1, passo) for x in range(0, w - lado + 1, passo)]
    return max(cantos, key=lambda c: gt[c[0]:c[0] + lado, c[1]:c[1] + lado].sum())


def mapa_erro(pred, gt):
    out = np.ones(gt.shape + (3,))
    out[pred & gt] = COR_VP
    out[pred & ~gt] = COR_FP
    out[~pred & gt] = COR_FN
    return out


def virgula(v):
    return f"{v:.3f}".replace(".", ",")


base05 = df[(df.condicao == "aprendido") & (df.limiar == 0.5)].sort_values("dif_dice")
escolhidas = [("pior × Cellpose", base05.iloc[0].id), ("mediana", base05.iloc[len(base05) // 2].id),
              ("melhor × Cellpose", base05.iloc[-1].id)]
titulos = ["imagem + GT", "marcador", "canal positivo", "probabilidade de saída", "aprendido",
           "aprendido + negativo", "Cellpose"]
fig, axes = plt.subplots(len(escolhidas), len(titulos), figsize=(2.6 * len(titulos), 3.3 * len(escolhidas)))
for linha, (rotulo, image_id) in enumerate(escolhidas):
    s, p = train_samples[image_id], PRED[image_id]
    gt = s["ground_truth"] > 0
    y, x = janela(gt)
    fatia = (slice(y, y + LADO), slice(x, x + LADO))
    contorno = find_boundaries(gt[fatia], mode="inner")
    img = s["image"][fatia].astype(float) / 255
    img[contorno] = 0
    probs = {c: p["condicoes"][c].astype(np.float32) for c in ("aprendido", "aprendido + negativo")}
    dices = {c: binary_metrics(v, gt)["dice"] for c, v in probs.items()}
    dices["Cellpose"] = binary_metrics(s["cellpose_segmentation"] > 0, gt)["dice"]
    paineis = [
        (img, None), (p["marcador"][fatia].astype(np.float32), "magma"), (p["pos"][fatia].astype(np.float32), "magma"),
        (probs["aprendido"][fatia], "magma"),
        (mapa_erro(probs["aprendido"][fatia] >= 0.5, gt[fatia]), None),
        (mapa_erro(probs["aprendido + negativo"][fatia] >= 0.5, gt[fatia]), None),
        (mapa_erro((s["cellpose_segmentation"] > 0)[fatia], gt[fatia]), None),
    ]
    for j, (painel, cmap) in enumerate(paineis):
        ax = axes[linha, j]
        ax.imshow(painel, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None, interpolation="nearest")
        if j == 3:
            ax.contour(gt[fatia], levels=[0.5], colors="white", linewidths=0.6)
        ax.set_xticks([]), ax.set_yticks([])
        topo = titulos[j] if linha == 0 else ""
        nome = {4: "aprendido", 5: "aprendido + negativo", 6: "Cellpose"}.get(j)
        ax.set_title((topo + "\n" if topo else "") + (f"Dice {virgula(dices[nome])}" if nome else ""), fontsize=9)
    axes[linha, 0].set_ylabel(f"{rotulo}\n{image_id[:12]}", fontsize=9)
fig.legend(handles=[Patch(color=COR_VP, label="acerto (núcleo)"), Patch(color=COR_FP, label="falso positivo"),
                    Patch(color=COR_FN, label="falso negativo")], loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Marcadores aprendidos e saída do ScribblePrompt (256², limiar 0,5); Dice da imagem inteira", fontsize=10)
fig.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

## 9. Salvar (opcional)

In [ ]:
if SALVAR_RESULTADOS:
    os.makedirs(OUTPUT_DIR, exist_ok=False)
    df.to_csv(os.path.join(OUTPUT_DIR, "por_imagem.csv.gz"), index=False)
    resumo.to_csv(os.path.join(OUTPUT_DIR, "resumo.csv"), index=False)
    diag.to_csv(os.path.join(OUTPUT_DIR, "diagnostico.csv"), index=False)
    perfil.to_csv(os.path.join(OUTPUT_DIR, "perfil.csv"), index=False)
    fig.savefig(os.path.join(OUTPUT_DIR, "figura_marcadores.png"), dpi=100)
    config = {
        "config": CONFIG.__dict__, "rgba_alpha": RGBA_ALPHA, "sp_input_size": SP_INPUT_SIZE, "tau": TAU,
        "tv_weight": TV_WEIGHT, "limiares": LIMIARES, "raio_negativo": RAIO_NEGATIVO,
        "folds": [{"fold": f.fold, "val_ids": f.val_ids, "best_epoch": f.best_epoch, "best_dice": f.best_dice,
                   "epochs_run": f.epochs_run} for f in folds],
        "procedencia": provenance(),
    }
    with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as fh:
        json.dump(config, fh, indent=2, ensure_ascii=False)
    print("Resultados em:", OUTPUT_DIR)
else:
    print("Nada salvo (SALVAR_RESULTADOS = False).")